In [ ]:
pip install scenedetect

In [ ]:
# die müssen vor dem Lauf der Skripte angepasst werden!!!

AUSGABEORDNER_UNFILTERED = "extracted_frames/lausitz/unfiltered" # wo die extrahierten Bilder ohne Inhalt-Filter gespeichert werden
VIDEO_PATH = "Die-Lausitz.mp4" 
AUSGABEORDNER_FILTERED = "extracted_frames/lausitz/filtered" # wo die Bilder nach der Inhalt-Filterung gespeichert werden

# Bilder-Extraktion
### Einstellung: Szenen ≤ 2s bekommen 1 Bild, >2s bekommen 3 Bilder

In [ ]:
import cv2
import os
from typing import List, Tuple
import numpy as np
from scenedetect import detect, ContentDetector


class MidframeExtractor:
    def __init__(self, 
                 short_scene_threshold: float = 2.0,
                 blur_threshold: float = 100.0,
                 output_dir: str = "extracted_frames"):
        """
        Initialisiert den Midframe Extractor
        
        Args:
            short_scene_threshold: Schwellenwert für kurze vs. lange Szenen in Sekunden
            blur_threshold: Schwellenwert für Unschärfeerkennung (höhere Werte = schärferes Bild erforderlich)
            output_dir: Ausgabeordner für die extrahierten Bilder
        """
        self.short_scene_threshold = short_scene_threshold
        self.blur_threshold = blur_threshold
        self.output_dir = output_dir
        
        # Erstelle Ausgabeordner falls nicht vorhanden
        os.makedirs(self.output_dir, exist_ok=True)
    
    def detect_scene_timestamps(self, video_path: str) -> List[Tuple[float, float]]:
        """
        Erkennt Szenen-Timestamps im Video
        
        Args:
            video_path: Pfad zur Videodatei
            
        Returns:
            Liste von (start_time, end_time) Tupeln in Sekunden
        """
        scene_list = detect(video_path, ContentDetector())
        
        # Konvertiere zu (start, end) Tupeln in Sekunden
        scene_timestamps = []
        for i, scene in enumerate(scene_list):
            start_time = scene[0].get_seconds()
            end_time = scene[1].get_seconds()
            scene_timestamps.append((start_time, end_time))
        
        return scene_timestamps
    
    def calculate_blur_score(self, image: np.ndarray) -> float:
        """
        Berechnet Unschärfe-Score eines Bildes mittels Laplacian Varianz
        
        Args:
            image: BGR Bild als numpy array
            
        Returns:
            Unschärfe-Score (höhere Werte = schärferes Bild)
        """
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        return cv2.Laplacian(gray, cv2.CV_64F).var()
    
    def extract_best_frame_from_timespan(self, cap: cv2.VideoCapture, 
                                       start_time: float, end_time: float,
                                       fps: float, sample_frames: int = 5) -> Tuple[np.ndarray, float]:
        """
        Extrahiert das schärfste Bild aus einem Zeitbereich
        
        Args:
            cap: OpenCV VideoCapture Objekt
            start_time: Startzeit in Sekunden
            end_time: Endzeit in Sekunden
            fps: Frames per Second des Videos
            sample_frames: Anzahl der zu sampelnden Frames für die Auswahl
            
        Returns:
            Tupel aus (best_frame, timestamp_of_best_frame)
        """
        # Berechne Frame-Nummern
        start_frame = int(start_time * fps)
        end_frame = int(end_time * fps)
        
        # Wähle Frames zum Sampeln aus
        total_frames = end_frame - start_frame + 1
        if total_frames <= sample_frames:
            # Wenn wenige Frames, nimm alle
            frames_to_check = list(range(start_frame, end_frame + 1))
        else:
            # Gleichmäßig verteilte Samples
            frames_to_check = np.linspace(start_frame, end_frame, sample_frames, dtype=int)
        
        best_frame = None
        best_blur_score = 0
        best_timestamp = start_time
        
        # Durchlaufe Sample-Frames und finde das schärfste
        for frame_num in frames_to_check:
            if frame_num >= cap.get(cv2.CAP_PROP_FRAME_COUNT):
                break
                
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
            ret, frame = cap.read()
            
            if not ret:
                continue
                
            blur_score = self.calculate_blur_score(frame)
            
            if blur_score > best_blur_score:
                best_blur_score = blur_score
                best_frame = frame.copy()
                best_timestamp = frame_num / fps
        
        return best_frame, best_timestamp
    
    def extract_frame_at_position(self, cap: cv2.VideoCapture, 
                                start_time: float, end_time: float, 
                                position: str, fps: float) -> Tuple[np.ndarray, float]:
        """
        Extrahiert das beste Frame an einer spezifischen Position (Anfang, Mitte, Ende)
        
        Args:
            cap: OpenCV VideoCapture Objekt
            start_time: Startzeit in Sekunden
            end_time: Endzeit in Sekunden
            position: 'start', 'middle', oder 'end'
            fps: Frames per Second des Videos
            
        Returns:
            Tupel aus (best_frame, timestamp_of_best_frame)
        """
        scene_duration = end_time - start_time
        
        if position == 'start':
            # Erste 20% der Szene oder max 1 Sekunde
            search_end = min(start_time + min(0.2 * scene_duration, 1.0), end_time)
            return self.extract_best_frame_from_timespan(cap, start_time, search_end, fps)
        
        elif position == 'middle':
            # Mittlere 20% der Szene
            middle_point = (start_time + end_time) / 2
            search_radius = min(0.1 * scene_duration, 0.5)  # Max 0.5 Sekunden Radius
            search_start = max(start_time, middle_point - search_radius)
            search_end = min(end_time, middle_point + search_radius)
            return self.extract_best_frame_from_timespan(cap, search_start, search_end, fps)
        
        elif position == 'end':
            # Letzte 20% der Szene oder max 1 Sekunde
            search_start = max(end_time - min(0.2 * scene_duration, 1.0), start_time)
            return self.extract_best_frame_from_timespan(cap, search_start, end_time, fps)
        
        else:
            raise ValueError(f"Unbekannte Position: {position}")
    
    def format_timestamp(self, timestamp: float) -> str:
        """
        Formatiert Timestamp für Dateinamen (HH-MM-SS-mmm)
        
        Args:
            timestamp: Zeit in Sekunden
            
        Returns:
            Formatierter String für Dateinamen
        """
        hours = int(timestamp // 3600)
        minutes = int((timestamp % 3600) // 60)
        seconds = int(timestamp % 60)
        milliseconds = int((timestamp % 1) * 1000)
        
        return f"{hours:02d}-{minutes:02d}-{seconds:02d}-{milliseconds:03d}"
    
    def extract_frames(self, video_path: str, scene_timestamps: List[Tuple[float, float]]) -> List[str]:
        """
        Extrahiert Frames basierend auf Szenen-Timestamps und neuen Regeln
        
        Args:
            video_path: Pfad zur Videodatei
            scene_timestamps: Liste von (start_time, end_time) Tupeln
            
        Returns:
            Liste der Pfade zu den extrahierten Bildern
        """
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise ValueError(f"Konnte Video nicht öffnen: {video_path}")
        
        fps = cap.get(cv2.CAP_PROP_FPS)
        video_name = os.path.splitext(os.path.basename(video_path))[0]
        
        extracted_files = []
        
        print(f"Verarbeite {len(scene_timestamps)} Szenen...")
        
        for i, (start_time, end_time) in enumerate(scene_timestamps):
            scene_duration = end_time - start_time
            
            # Regel 1: Szenen ≤ 2s -> 1 Bild (Mitte der Szene)
            if scene_duration <= self.short_scene_threshold:
                frame, timestamp = self.extract_best_frame_from_timespan(
                    cap, start_time, end_time, fps)
                
                if frame is not None:
                    blur_score = self.calculate_blur_score(frame)
                    if blur_score >= self.blur_threshold:
                        timestamp_str = self.format_timestamp(timestamp)
                        filename = f"{video_name}_scene{i+1:03d}_mid_{timestamp_str}.jpg"
                        filepath = os.path.join(self.output_dir, filename)
                        
                        cv2.imwrite(filepath, frame)
                        extracted_files.append(filepath)
                        print(f"Szene {i+1}: {scene_duration:.2f}s - 1 Bild extrahiert bei {timestamp:.2f}s (Schärfe: {blur_score:.1f})")
                    else:
                        print(f"Szene {i+1}: {scene_duration:.2f}s - Bild zu unscharf (Schärfe: {blur_score:.1f})")
            
            # Regel 2: Szenen > 2s -> 3 Bilder (Anfang, Mitte, Ende)
            else:
                positions = ['start', 'middle', 'end']
                position_names = ['start', 'mid', 'end']
                images_extracted = 0
                
                for pos, pos_name in zip(positions, position_names):
                    frame, timestamp = self.extract_frame_at_position(
                        cap, start_time, end_time, pos, fps)
                    
                    if frame is not None:
                        blur_score = self.calculate_blur_score(frame)
                        if blur_score >= self.blur_threshold:
                            timestamp_str = self.format_timestamp(timestamp)
                            filename = f"{video_name}_scene{i+1:03d}_{pos_name}_{timestamp_str}.jpg"
                            filepath = os.path.join(self.output_dir, filename)
                            
                            cv2.imwrite(filepath, frame)
                            extracted_files.append(filepath)
                            images_extracted += 1
                
                print(f"Szene {i+1}: {scene_duration:.2f}s - {images_extracted}/3 Bilder extrahiert")
        
        cap.release()
        return extracted_files
    
    def calculate_scene_statistics(self, scene_timestamps: List[Tuple[float, float]]) -> None:
        """
        Berechnet und zeigt Statistiken über die Szenen an
        
        Args:
            scene_timestamps: Liste von (start_time, end_time) Tupeln
        """
        durations = [end - start for start, end in scene_timestamps]
        
        short_scenes = sum(1 for d in durations if d <= self.short_scene_threshold)
        long_scenes = len(durations) - short_scenes
        
        estimated_images = short_scenes * 1 + long_scenes * 3
        
        print(f"\n=== Szenen-Statistiken ===")
        print(f"Gesamtanzahl Szenen: {len(scene_timestamps)}")
        print(f"Kurze Szenen (≤{self.short_scene_threshold}s): {short_scenes}")
        print(f"Lange Szenen (>{self.short_scene_threshold}s): {long_scenes}")
        print(f"Geschätzte Anzahl Bilder: {estimated_images}")
        print(f"Durchschnittliche Szenenlänge: {np.mean(durations):.2f}s")
        print(f"Median Szenenlänge: {np.median(durations):.2f}s")
    
    def process_video(self, video_path: str) -> List[str]:
        """
        Kompletter Pipeline-Prozess: Szenendetection + Frame-Extraktion
        
        Args:
            video_path: Pfad zur Videodatei
            
        Returns:
            Liste der Pfade zu den extrahierten Bildern
        """
        print(f"Starte Szenendetection für: {video_path}")
        scene_timestamps = self.detect_scene_timestamps(video_path)
        
        print(f"Gefundene Szenen: {len(scene_timestamps)}")
        print(f"Einstellungen: short_scene_threshold={self.short_scene_threshold}s, "
              f"blur_threshold={self.blur_threshold}")
        
        # Zeige Statistiken
        self.calculate_scene_statistics(scene_timestamps)
        
        extracted_files = self.extract_frames(video_path, scene_timestamps)
        
        print(f"\nExtraktion abgeschlossen!")
        print(f"Extrahierte Bilder: {len(extracted_files)}")
        print(f"Gespeichert in: {self.output_dir}")
        
        return extracted_files


## Längenverteilung plotten lassen

In [ ]:
def detect_scene_timestamps (video_path, screenshot_path = None): 
    scene_timestamps = detect(video_path, ContentDetector())
    if screenshot_path != None: 
        videostream = open_video(video_path)
        images = save_images(scene_list=scene_timestamps,video=videostream,output_dir=screenshot_path,num_images=1)
        return (scene_timestamps, images)
    else:
        return scene_timestamps

scene_ts = detect_scene_timestamps(video_path=VIDEO_PATH)

In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np

def plot_scene_length_histogram(durations, bins=None, bin_size=None,
                                title="Längenverteilung der Szenen",
                                xlabel="Dauer (Sekunden)", ylabel="Anzahl Szenen",
                                save_path=None, show=True):
    """
    Zeichnet ein Histogramm der Szenendauern.

    Parameter
    ---------
    durations : list[float]
        Szenendauern in Sekunden.
    bins : int | sequence[float] | None
        Wie in matplotlib: Anzahl der Bins oder explizite Bin-Kanten.
        Falls None und bin_size ist gesetzt, werden die Bins aus bin_size berechnet.
        Falls beides None, wird eine heuristische Bin-Anzahl verwendet.
    bin_size : float | None
        Bin-Breite in Sekunden (z. B. 2.0 für 2s).
        Wird ignoriert, wenn 'bins' bereits gesetzt ist.
    title, xlabel, ylabel : str
        Beschriftungen und Titel.
    save_path : str | None
        Pfad zum Speichern (z. B. 'hist_szenen.png'). Wenn None, wird nicht gespeichert.
    show : bool
        Wenn True, wird die Grafik angezeigt.

    Returns
    -------
    fig, ax : matplotlib.figure.Figure, matplotlib.axes.Axes
        Figure- und Axes-Objekte zur weiteren Verwendung.
    """
    if not durations:
        raise ValueError("Die Liste 'durations' ist leer. Nichts zu plotten.")

    durations = np.asarray(durations, dtype=float)

    # Falls keine Bins angegeben sind, aus bin_size ableiten oder heuristisch bestimmen
    if bins is None:
        if bin_size is not None and bin_size > 0:
            max_val = durations.max()
            # Kanten von 0 bis >= max_val in Schritten von bin_size
            edges = np.arange(0, math.ceil(max_val / bin_size) * bin_size + bin_size, bin_size)
            bins = edges
        else:
            # Heuristik: Freedman–Diaconis-Regel als gute Default-Wahl
            q75, q25 = np.percentile(durations, [75, 25])
            iqr = max(q75 - q25, 1e-9)
            bin_width = 2 * iqr * (len(durations) ** (-1/3))
            if bin_width <= 0:
                bin_width = max(durations.max() - durations.min(), 1.0) / 10.0
            n_bins = max(5, int(math.ceil((durations.max() - durations.min()) / bin_width)))
            bins = n_bins

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.hist(durations, bins=bins, edgecolor="black", color="#4C78A8")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(True, axis="y", linestyle=":", alpha=0.5)

    # Übersichtliche x-Achse für kurze Szenen
    if durations.max() <= 60:
        ax.set_xticks(np.arange(0, max(10, int(math.ceil(durations.max())) + 1), 2))

    fig.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=150)
    if show:
        plt.show()

    return fig, ax

def scene_length_distribution(scene_ts, sort=True):
    """
    Berechnet die Längen aller Szenen in Sekunden aus einer Liste von (start, end)-FrameTimecodes.
    Gibt eine Liste von Floats zurück.
    """
    if not scene_ts:
        return []

    durations = []
    for start, end in scene_ts:
        delta = end - start
        if hasattr(delta, "get_seconds"):
            durations.append(delta.get_seconds())
        else:
            s = start.get_seconds() if hasattr(start, "get_seconds") else float(start)
            e = end.get_seconds() if hasattr(end, "get_seconds") else float(end)
            durations.append(e - s)

    if sort:
        durations.sort()
    return durations
    

In [ ]:
# Dauerliste
durations = scene_length_distribution(scene_ts, sort=False)

# Histogramm mit Bin-Größe 2 Sekunden
plot_scene_length_histogram(durations, bin_size=2.0,
                            title="Szenendauern (Bins = 2 s)",
                            save_path=None, show=True)

## Bilder-Extraktion ausführen lassen

In [ ]:
if __name__ == "__main__":
    # Flexibel konfigurierbar
    extractor = MidframeExtractor(
        short_scene_threshold=2.0,       # Szenen ≤ 2s bekommen 1 Bild, >2s bekommen 3 Bilder
        blur_threshold=30.0,            # Mindest-Schärfe-Score WICHTIGER PARAMETER -> hat Einfluss auf die QUalität der extrahirten Bilder
        output_dir=AUSGABEORDNER_UNFILTERED    # Ausgabeordner
    )
    
    # Video verarbeiten
    video_path = VIDEO_PATH  # Pfad zu eurer Videodatei
    extracted_files = extractor.process_video(video_path)
    
    print(f"\nExtrahierte Dateien:")
    for file in extracted_files[:10]:  # Zeige erste 10 als Beispiel
        print(f"  {file}")
    if len(extracted_files) > 10:
        print(f"  ... und {len(extracted_files) - 10} weitere")

## Filter von Claus - Bilder mit gleichem oder ähnlichem Inhalt werden gelöscht

In [ ]:
##################################################################
# Projektarbeit Audiodeskription
#
# Modul zur Reduktion ähnlicher Bilder in einem Ordner
#
# Gruppe: Adriana Klaja, Claus-Peter Koch
#
# Version: 0.1.0.0  - 25.08.2025
#
# !!!!! Testversion
# >> Liest aus einem Ordner, z.B. mit Bildern 1x/Sek und kopiert ausgewählte Bilder
# >> Erstellt ein neues Verzeichnis, in dem nur wenige Bilder verbleiben, die den vorherigen deutlich abweichen
#
# ##################################################################


# Bildreduktion2.py  — v2 (dHash + pHash + SSIM, Hysterese, Mindestabstand)
# Bildreduktion2.py --in FilmBilder --out FilmBilderGefiltert --min-gap 1 --dh-soft 12 --ph-soft 40 --ssim-soft 0.998 --ph-hard 96 --ssim-hard 0.985


import cv2, numpy as np, re, shutil, sys
from pathlib import Path
try:
    from skimage.metrics import structural_similarity as ssim
except Exception:
    ssim = None

#print(">> Bildreduktion.py v2 — aktiv:", __file__)

def natural_key(s): return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', s)]
def hamming(a, b): return (a ^ b).bit_count()

def dhash64(img_bgr, hash_size=8):
    g = cv2.cvtColor(cv2.resize(img_bgr, (hash_size+1, hash_size)), cv2.COLOR_BGR2GRAY)
    diff = g[:, 1:] > g[:, :-1]
    bits = 0
    for i, v in enumerate(diff.flatten()):
        if v: bits |= 1 << i
    return bits  # 64-bit

def phash256(img_bgr):
    g = cv2.cvtColor(cv2.resize(img_bgr, (32, 32)), cv2.COLOR_BGR2GRAY).astype(np.float32)
    d = cv2.dct(g)[:16, :16]
    d[0,0] = 0.0
    med = float(np.median(d))
    bits = 0; idx = 0
    for y in range(16):
        for x in range(16):
            if d[y, x] > med: bits |= 1 << idx
            idx += 1
    return bits  # 256-bit

def ssim_gray_small(a_gray, b_gray, target_max=512):
    if ssim is None: return None
    ha, wa = a_gray.shape[:2]
    scale = min(1.0, target_max / max(ha, wa))
    if scale < 1.0:
        a_gray = cv2.resize(a_gray, (int(wa*scale), int(ha*scale)), interpolation=cv2.INTER_AREA)
        b_gray = cv2.resize(b_gray, (int(wa*scale), int(ha*scale)), interpolation=cv2.INTER_AREA)
    return ssim(a_gray, b_gray)

def dedup_stream(
    in_dir, out_dir,
    dh_soft=12, ph_soft=40, ssim_soft=0.998,   # ähnlich → verwerfen
    ph_hard=96, ssim_hard=0.985,               # starker Wechsel → trotz min_gap behalten
    min_gap=5,                                  # Mindestabstand (Frames/Sekunden)
    dry_run=False
):
    in_dir, out_dir = Path(in_dir), Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    files = []
    for ext in ("*.jpg","*.jpeg","*.png","*.bmp","*.webp"):
        files.extend(in_dir.glob(ext))
    files.sort(key=lambda p: natural_key(p.name))

    anchor_dh = anchor_ph = None
    anchor_gray = None
    since_keep = 10**9
    kept = 0

    for p in files:
        img = cv2.imread(str(p), cv2.IMREAD_COLOR)
        if img is None: continue
        g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        dh = dhash64(img)
        ph = phash256(img)

        if anchor_dh is None:
            do_keep = True
            strong_change = True
            similar = False
        else:
            d_dh = hamming(dh, anchor_dh)       # 0..64
            d_ph = hamming(ph, anchor_ph)       # 0..256
            s = ssim_gray_small(anchor_gray, g) if ssim else None

            similar = ((d_dh < dh_soft and d_ph < ph_soft) or
                       (s is not None and s >= ssim_soft))
            strong_change = ((d_ph >= ph_hard) or
                             (s is not None and s <= ssim_hard))

            if since_keep < min_gap and not strong_change:
                do_keep = False
            else:
                do_keep = not similar

        if do_keep:
            if not dry_run:
                shutil.copy2(p, out_dir / p.name)
            anchor_dh, anchor_ph, anchor_gray = dh, ph, g
            kept += 1
            since_keep = 0
        else:
            since_keep += 1

    print(f"Fertig. Behalten: {kept} von {len(files)} (min_gap={min_gap}).")



## Filter von Claus ausführen lassen

In [ ]:
# Default-Werte für Jupyter statt argparse
defaults = dict(
    extracted_frames=AUSGABEORDNER_UNFILTERED,        # Pflichtparameter in CLI-Version
    extracted_frames_reduced=AUSGABEORDNER_FILTERED, # Pflichtparameter in CLI-Version
    dh_soft=30, #12
    ph_soft=70, #40
    ssim_soft=0.8, #0.985
    ph_hard=150, #96
    ssim_hard=0.8, #0.995
    min_gap=1,
    dry_run=False,
)

# Aliase (falls man sie nutzen will, wie bei CLI)
ham = None    # z.B. 15, falls man explizit setzen will
ssim = None   # z.B. 0.995

if ham is not None and defaults["dh_soft"] == 12:
    defaults["dh_soft"] = ham
if ssim is not None and defaults["ssim_soft"] == 0.998:
    defaults["ssim_soft"] = ssim

# Funktionsaufruf mit den Defaults
dedup_stream(
    defaults["extracted_frames"],
    defaults["extracted_frames_reduced"],
    dh_soft=defaults["dh_soft"],
    ph_soft=defaults["ph_soft"],
    ssim_soft=defaults["ssim_soft"],
    ph_hard=defaults["ph_hard"],
    ssim_hard=defaults["ssim_hard"],
    min_gap=defaults["min_gap"],
    dry_run=defaults["dry_run"],
)



## CSV erstellen

In [ ]:
import os
import csv
import re
from pathlib import Path

def extract_info_from_filename(filename):
    """
    Extrahiert Informationen aus dem Dateinamen.
    Erwartet Format: Name_sceneXXX_wann_timestamp.jpg
    """
    # Entferne die .jpg Endung
    base_name = filename.replace('.jpg', '')
    
    # Regex Pattern für die verschiedenen Teile
    # Pattern erklärt: (.+?)_(scene\d+)_(\w+)_(.+)
    # (.+?) = Name (non-greedy)
    # (scene\d+) = scene + Zahlen
    # (\w+) = wann (Wörter)
    # (.+) = timestamp (Rest)
    pattern = r'(.+?)_(scene\d+)_(\w+)_(.+)'
    
    match = re.match(pattern, base_name)
    
    if match:
        name, scene, wann, timestamp = match.groups()
        return {
            'dateiname': filename,
            'szenennummer': scene,
            'wann': wann,
            'timestamp': timestamp
        }
    else:
        # Falls das Pattern nicht passt, trotzdem Dateiname erfassen
        return {
            'dateiname': filename,
            'szenennummer': 'unbekannt',
            'wann': 'unbekannt',
            'timestamp': 'unbekannt'
        }

def create_csv_from_images(directory_path, output_csv='hamster_bilder.csv'):
    """
    Erstellt eine CSV-Datei aus allen JPG-Dateien im angegebenen Verzeichnis.
    Ignoriert Unterordner.
    """
    directory = Path(directory_path)
    
    if not directory.exists():
        print(f"Fehler: Verzeichnis '{directory_path}' existiert nicht.")
        return
    
    # Sammle alle .jpg Dateien (nur im Hauptverzeichnis, keine Unterordner)
    jpg_files = [f for f in directory.iterdir() if f.is_file() and f.suffix.lower() == '.jpg']
    
    if not jpg_files:
        print(f"Keine JPG-Dateien im Verzeichnis '{directory_path}' gefunden.")
        return
    
    print(f"{len(jpg_files)} JPG-Dateien gefunden.")
    
    # Erstelle CSV
    with open(output_csv, 'w', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['Dateiname', 'Szenennummer', 'wann', 'timestamp']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        
        # Schreibe Header
        writer.writeheader()
        
        # Verarbeite jede JPG-Datei
        for jpg_file in sorted(jpg_files):
            info = extract_info_from_filename(jpg_file.name)
            
            writer.writerow({
                'Dateiname': info['dateiname'],
                'Szenennummer': info['szenennummer'],
                'wann': info['wann'],
                'timestamp': info['timestamp']
            })
    
    print(f"CSV-Datei '{output_csv}' wurde erfolgreich erstellt.")
    print(f"Verarbeitete Dateien: {len(jpg_files)}")


In [ ]:
def main():
    verzeichnis = "xxx" # von welchen Bildern soll die csv erstellt werden -> entweder AUSGABEORDNER_UNFILTERED oder AUSGABEORDNER_FILTERED
    output_datei = "xxx.csv" # Name für die csv-Datei
    
    print(f"Verarbeite JPG-Dateien aus: {verzeichnis}")
    print(f"Ausgabe CSV: {output_datei}")
    print("-" * 50)
    
    create_csv_from_images(verzeichnis, output_datei)
    
    # Zeige ein paar Beispielzeilen der erstellten CSV
    if os.path.exists(output_datei):
        print("\nErste 5 Zeilen der CSV:")
        print("-" * 50)
        with open(output_datei, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                if i < 6:  # Header + 5 Zeilen
                    print(line.strip())
                else:
                    break

In [ ]:
if __name__ == "__main__":
    main()